<a href="https://colab.research.google.com/github/8009678200/ML.github.io/blob/main/hd2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install Streamlit and pyngrok
!{sys.executable} -m pip install streamlit pyngrok

/bin/bash: line 1: {sys.executable}: command not found


Next, I'll create the `app.py` file which contains the Streamlit application code. This code includes a function to manually parse the CSV file (without using pandas), and then implements the logic for sorting and pagination, and finally displays the data using `st.table`.

In [10]:
%%writefile app.py

import streamlit as st
import csv
import math

# Function to manually parse CSV without pandas
def parse_csv_data(filepath):
    data = []
    headers = []
    try:
        with open(filepath, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            headers = next(reader) # First row is headers
            for row in reader:
                if row: # Ensure row is not empty
                    data.append(row)
    except FileNotFoundError:
        st.error(f"File not found: {filepath}")
        return [], []
    except Exception as e:
        st.error(f"Error parsing CSV: {e}")
        return [], []
    return headers, data

# Function to convert specified categorical columns to numerical
def convert_categorical_to_numerical(headers, data, categorical_cols):
    # Create a copy to avoid modifying original data during iteration issues
    processed_data = [list(row) for row in data]

    for col_name in categorical_cols:
        try:
            col_idx = headers.index(col_name)
            unique_values = sorted(list(set(row[col_idx] for row in processed_data if len(row) > col_idx)))
            mapping = {value: i for i, value in enumerate(unique_values)}

            for i, row in enumerate(processed_data):
                if len(row) > col_idx:
                    original_value = row[col_idx]
                    # Assign numerical value; handle unknown values by assigning -1 or a unique new number
                    processed_data[i][col_idx] = mapping.get(original_value, -1) # -1 for unknown values
        except ValueError:
            # Column not found in headers, skip conversion for this column
            st.warning(f"Categorical column '{col_name}' not found in dataset headers. Skipping conversion.")
    return processed_data

# Function to calculate mean
def calculate_mean(data_list):
    if not data_list:
        return None
    total = 0
    count = 0
    for x in data_list:
        try:
            total += float(x)
            count += 1
        except ValueError:
            continue # Skip non-numeric values
    return total / count if count > 0 else None

# Function to calculate median
def calculate_median(data_list):
    numeric_data = []
    for x in data_list:
        try:
            numeric_data.append(float(x))
        except ValueError:
            continue
    if not numeric_data:
        return None
    sorted_data = sorted(numeric_data)
    n = len(sorted_data)
    if n % 2 == 1:
        return sorted_data[n // 2]
    else:
        mid1 = sorted_data[n // 2 - 1]
        mid2 = sorted_data[n // 2]
        return (mid1 + mid2) / 2

# Function to calculate mode
def calculate_mode(data_list):
    counts = {}
    for x in data_list:
        # Mode can be calculated for both numeric and non-numeric
        counts[x] = counts.get(x, 0) + 1
    if not counts:
        return None
    max_count = 0
    modes = []
    for value, count in counts.items():
        if count > max_count:
            max_count = count
            modes = [value]
        elif count == max_count and value not in modes:
            modes.append(value)
    return modes # Return a list because there can be multiple modes

# Function to calculate min
def calculate_min(data_list):
    if not data_list:
        return None
    min_val = None
    for x in data_list:
        try:
            val = float(x)
            if min_val is None or val < min_val:
                min_val = val
        except ValueError:
            continue
    return min_val

# Function to calculate max
def calculate_max(data_list):
    if not data_list:
        return None
    max_val = None
    for x in data_list:
        try:
            val = float(x)
            if max_val is None or val > max_val:
                max_val = val
        except ValueError:
            continue
    return max_val

# Function to calculate standard deviation (sample standard deviation)
def calculate_std_dev(data_list):
    numeric_data = []
    for x in data_list:
        try:
            numeric_data.append(float(x))
        except ValueError:
            continue
    if len(numeric_data) < 2: # Need at least two points for sample std dev
        return 0.0 if len(numeric_data) == 1 else None

    mean_val = calculate_mean(numeric_data)
    if mean_val is None:
        return None

    sum_sq_diff = 0
    for x in numeric_data:
        sum_sq_diff += (x - mean_val) ** 2

    # Using sample standard deviation (n-1 degrees of freedom)
    return math.sqrt(sum_sq_diff / (len(numeric_data) - 1))


# Streamlit App
st.title("Heart Disease Prediction Dataset Viewer")

# File path
FILE_PATH = "/content/Heart_Disease_Prediction.csv"

headers, raw_data = parse_csv_data(FILE_PATH)

if not headers or not raw_data:
    st.warning("No data to display or an error occurred during parsing.")
else:
    st.write(f"Dataset loaded: `{FILE_PATH}`")
    st.write(f"Total records: {len(raw_data)}")

    # Specify categorical columns to convert
    categorical_columns_to_convert = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

    # Convert categorical data to numerical
    processed_data = convert_categorical_to_numerical(headers, raw_data, categorical_columns_to_convert)

    # --- Sorting Options ---
    st.sidebar.header("Sorting Options")
    sort_column = st.radio("Sort by column", options=headers)
    sort_ascending = st.checkbox("Ascending order", value=True)

    sorted_data = []
    if sort_column:
        # To sort, we need the index of the column
        try:
            sort_col_idx = headers.index(sort_column)
            # Create a sorting key function
            def get_sort_key_for_list(row_list):
                value = row_list[sort_col_idx]
                try:
                    return float(value) # Try to sort numerically if possible
                except ValueError:
                    return value # Otherwise, sort as string
            sorted_data = sorted(processed_data, key=get_sort_key_for_list, reverse=not sort_ascending)
        except ValueError:
            sorted_data = processed_data # Fallback if column not found
    else:
        sorted_data = processed_data # No sort column selected


    # --- Pagination Options ---
    st.sidebar.header("Pagination Options")
    items_per_page_options = [10, 25, 50, 100]
    items_per_page = st.selectbox("Items per page", options=items_per_page_options, index=0)

    total_pages = math.ceil(len(sorted_data) / items_per_page)

    # Using session state for current page
    if 'current_page' not in st.session_state:
        st.session_state.current_page = 1

    col1, col2, col3 = st.sidebar.columns([1,2,1])
    with col1:
        if st.button("Previous Page", key="prev_page"):
            if st.session_state.current_page > 1:
                st.session_state.current_page -= 1
    with col3:
        if st.button("Next Page", key="next_page"):
            if st.session_state.current_page < total_pages:
                st.session_state.current_page += 1
    with col2:
        st.write(f"Page {st.session_state.current_page} of {total_pages}")


    start_idx = (st.session_state.current_page - 1) * items_per_page
    end_idx = start_idx + items_per_page
    paginated_data = sorted_data[start_idx:end_idx]

    # --- Display Data ---
    st.header("Dataset Preview")
    st.table([headers] + paginated_data)

    # --- Summary Statistics Options ---
    st.sidebar.header("Summary Statistics")
    selected_stat_column = st.sidebar.selectbox("Select column for statistics", options=[''] + headers, index=0)

    if selected_stat_column:
        st.subheader(f"Summary Statistics for '{selected_stat_column}'")
        col_idx = headers.index(selected_stat_column)
        column_data = [row[col_idx] for row in processed_data if len(row) > col_idx]

        # Filter out non-numeric values for calculations that require them
        numeric_column_data = []
        for x in column_data:
            try:
                numeric_column_data.append(float(x))
            except ValueError:
                continue

        mean_val = calculate_mean(column_data)
        median_val = calculate_median(column_data)
        mode_val = calculate_mode(column_data)
        min_val = calculate_min(column_data)
        max_val = calculate_max(column_data)
        std_dev_val = calculate_std_dev(column_data)

        stats_data = {
            "Statistic": ["Mean", "Median", "Mode", "Min", "Max", "Standard Deviation", "Count"],
            "Value": [
                f"{mean_val:.2f}" if mean_val is not None else "N/A",
                f"{median_val:.2f}" if median_val is not None else "N/A",
                str(mode_val) if mode_val is not None else "N/A",
                f"{min_val:.2f}" if min_val is not None else "N/A",
                f"{max_val:.2f}" if max_val is not None else "N/A",
                f"{std_dev_val:.2f}" if std_dev_val is not None else "N/A",
                len(numeric_column_data)
            ]
        }
        st.table(stats_data)


Overwriting app.py


Finally, I'll run the Streamlit app. After execution, a public URL will be generated. Click on that URL to interact with the Streamlit app.

In [11]:
import subprocess
import time

!pip install pyngrok streamlit

from pyngrok import ngrok

# Terminate any running Streamlit processes
!pkill -f streamlit

# Start Streamlit app in the background
streamlit_process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.enableCORS=False",
    "--server.enableXsrfProtection=False",
    "--browser.gatherUsageStats=False"
])

time.sleep(5) # Give Streamlit a moment to start

# Authenticate ngrok. Replace 'YOUR_NGROK_AUTHTOKEN' with your actual token if needed.
ngrok.set_auth_token("3FffIo4dyhAPhgAKHzBgJmWqW9H_81M6pUJLLN58mBeBtbWRL")

# Open a ngrok tunnel to the Streamlit port (8501)
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

Streamlit App URL: NgrokTunnel: "https://cubicle-humility-pried.ngrok-free.dev" -> "http://localhost:8501"
